# Phase 2 (quarterly) — Node feature engineering
Parallel to `phase2_node_features.ipynb` but at quarterly frequency. Loads `compustat_CIQ_quarterly.csv`, applies the same quality filters as the annual driver, renames quarterly columns to annual names (snapshots map directly; flow variables become **TTM** trailing 12-month sums so each quarter-end produces an annualized comparable). Then attaches CRSP trailing market features and FRED macro controls, saves `node_features_quarterly.parquet`, and validates with an Enron trajectory spot-check.

The annual `node_features_raw.parquet` is **not** modified.

In [ ]:
import sys, time
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[1]
sys.path.append(str(PROJECT_ROOT))

import numpy as np
import pandas as pd

from node_processing import (
    levereage_solvency as lev,
    profitability as prof,
    liquidity as liq,
    activity_efficiency as act,
    growth as grow,
    cash_flow as cf,
    composite_scores as cs,
)

t0 = time.time()
def stamp(msg): print(f'[{time.time()-t0:6.1f}s] {msg}', flush=True)

## 1. Load + filter quarterly Compustat
Apply the same quality filters as the annual driver (`datafmt=='STD'`, `consol=='C'`, `curcdq=='USD'`), deduplicate `gvkey + datadate`, drop rows where `atq`, `saleq`, AND `revtq` are all null.

Loading only the columns we need keeps memory in check (the raw file has 679 columns, we use ~38).

In [ ]:
QCOLS = [
    # IDs / filters
    'gvkey', 'datadate', 'conm', 'sic', 'fyearq', 'fqtr',
    'datafmt', 'consol', 'curcdq', 'costat', 'indfmt',
    # Balance sheet snapshot (quarter-end)
    'atq', 'actq', 'lctq', 'ltq', 'cheq', 'wcapq',
    'dlcq', 'dlttq', 'ceqq', 'req', 'invtq', 'rectq',
    # Income statement, single-quarter flow
    'niq', 'oiadpq', 'oibdpq', 'piq', 'xintq', 'cogsq',
    'saleq', 'revtq', 'dpq', 'xrdq', 'xsgaq',
    # YTD cash flow (must be diffed within fiscal year to get a quarterly flow)
    'oancfy', 'capxy',
    # Market snapshot
    'cshoq', 'prccq', 'mkvaltq',
]
stamp('loading compustat quarterly...')
dq = pd.read_csv(
    PROJECT_ROOT / 'data/raw/compustat/compustat_CIQ_quarterly.csv',
    usecols=QCOLS,
    dtype={'gvkey': 'Int64', 'sic': 'Int64', 'fyearq': 'Int64', 'fqtr': 'Int64'},
    low_memory=False,
)
stamp(f'raw quarterly: {len(dq):,} rows × {dq.shape[1]} cols')

In [ ]:
before = len(dq)
dq = dq[(dq['datafmt'] == 'STD') & (dq['consol'] == 'C') & (dq['curcdq'] == 'USD')]
print(f'STD/C/USD filter: {len(dq):,} (-{before-len(dq):,})')

dq['datadate'] = pd.to_datetime(dq['datadate'])
before = len(dq)
dq = dq.drop_duplicates(['gvkey', 'datadate'], keep='last')
print(f'dedup gvkey+datadate: {len(dq):,} (-{before-len(dq):,})')

before = len(dq)
empty = dq['atq'].isna() & dq['saleq'].isna() & dq['revtq'].isna()
dq = dq[~empty]
print(f'drop empty (atq+saleq+revtq all null): {len(dq):,} (-{before-len(dq):,})')

dq = dq.sort_values(['gvkey', 'datadate']).reset_index(drop=True)
print(f'final: {len(dq):,} rows × {dq.shape[1]} cols')

## 2. Map quarterly columns → annual names
The annual `node_processing/` modules consume these annual columns:

| Module | Inputs |
|---|---|
| leverage_solvency | `dltt, dlc, at, ceq, ebitda, xint` |
| profitability    | `ni, at, ceq, ebitda, sale, gp, oiadp` |
| liquidity        | `act, lct, che, rect, wcap, at` |
| size_features    | `at, sale, csho, prcc_f, emp` |
| activity         | `sale, at, rect, cogs, invt` |
| growth           | `sale, at, emp` |
| cash_flow        | `oancf, capx, at` |
| composite_scores | `act, lct, re, ebit, csho, prcc_f, lt, sale, at, wcap, ceq, sic` |

### Mapping strategy
Compustat quarterly distinguishes:
- **Snapshot** point-in-time fields (`q` suffix): balance-sheet items at quarter end. Map directly.
- **Single-quarter flow** fields (`q` suffix on income statement items): values for that quarter only. Convert to **TTM = trailing 4-quarter sum** so the resulting feature is comparable to an annual figure.
- **YTD cumulative flow** fields (`y` suffix; `oancfy`, `capxy`): cumulative since the start of the fiscal year. Difference within `(gvkey, fyearq)` to recover the quarterly flow, then take TTM.

### Annual columns with NO quarterly equivalent
- `emp` — employee count is annual-only in Compustat. `emp` and `emp_growth` features are NaN at quarterly frequency.
- `gp` — no `gpq`. Derived as `sale_ttm - cogs_ttm`.
- `ebit` — no `ebitq`. Falls through the EBIT fallback chain to `oiadp_ttm` or `pi_ttm + xint_ttm`.

In [ ]:
# Recover quarterly OCF / CAPX from YTD via diff within (gvkey, fyearq)
def ytd_to_q(ytd_col, fyear_col_name='fyearq'):
    grp = dq.groupby(['gvkey', fyear_col_name], sort=False)[ytd_col]
    diffed = grp.diff()
    first_in_year = grp.cumcount() == 0
    return diffed.where(~first_in_year, dq[ytd_col])

dq['oancfq_built'] = ytd_to_q('oancfy')
dq['capxq_built']  = ytd_to_q('capxy')

# TTM: trailing 4-quarter sum, requires 4 consecutive observations
def ttm(col):
    return (dq.groupby('gvkey', sort=False)[col]
              .rolling(window=4, min_periods=4).sum()
              .reset_index(level=0, drop=True))

flow_q_cols = ['niq', 'oiadpq', 'oibdpq', 'piq', 'xintq', 'cogsq',
               'saleq', 'revtq', 'dpq', 'xrdq', 'xsgaq',
               'oancfq_built', 'capxq_built']
for c in flow_q_cols:
    dq[c + '_ttm'] = ttm(c)
print('built TTM cols for', len(flow_q_cols), 'flow variables')

In [ ]:
RENAME = {
    # snapshots q -> annual
    'atq': 'at', 'actq': 'act', 'lctq': 'lct', 'ltq': 'lt',
    'cheq': 'che', 'wcapq': 'wcap',
    'dlcq': 'dlc', 'dlttq': 'dltt',
    'ceqq': 'ceq', 'req': 're',
    'invtq': 'invt', 'rectq': 'rect',
    'cshoq': 'csho', 'prccq': 'prcc_f', 'mkvaltq': 'mkvalt',
    # TTM flows -> annual flow names
    'niq_ttm':         'ni',
    'oiadpq_ttm':      'oiadp',
    'oibdpq_ttm':      'ebitda',     # oibdp ≈ EBITDA
    'piq_ttm':         'pi',
    'xintq_ttm':       'xint',
    'cogsq_ttm':       'cogs',
    'saleq_ttm':       'sale',
    'revtq_ttm':       'revt',
    'dpq_ttm':         'dp',
    'xrdq_ttm':        'xrd',
    'xsgaq_ttm':       'xsga',
    'oancfq_built_ttm':'oancf',
    'capxq_built_ttm': 'capx',
}
df = dq.rename(columns=RENAME).copy()

# Annual `gp` derived; `ebit` not present (falls through fallback); `emp` unavailable
df['gp']   = df['sale'] - df['cogs']
df['ebit'] = np.nan
df['emp']  = np.nan
print('renamed frame shape:', df.shape)

## 3. Compute features module-by-module
Identical formulas to the annual driver — same `node_processing/` modules, same column names (courtesy of the rename in step 2). Three module bugs from the annual driver are still inlined.

In [ ]:
ZERO_TO_NAN = ['at', 'ceq', 'sale', 'lct', 'lt', 'xint', 'rect', 'invt', 'cogs']
for c in ZERO_TO_NAN:
    df[c] = df[c].replace(0, np.nan)

feat = df[['gvkey', 'datadate', 'conm', 'sic', 'fyearq', 'fqtr']].copy()
feat['node_type'] = np.where(df['sic'].between(6000, 6999), 'financial', 'nonfinancial')

# Leverage / solvency
total_debt = df['dltt'].add(df['dlc'], fill_value=0)
total_debt = total_debt.where(total_debt != 0, np.nan)
feat['total_debt']        = total_debt
feat['debt_to_assets']    = lev.debt_to_assets(total_debt, df['at'])
feat['debt_to_equity']    = lev.debt_to_equity(total_debt, df['ceq'])
feat['lt_debt_ratio']     = df['dltt'] / df['at']
feat['st_debt_ratio']     = lev.st_debt_ration(df['dlc'], df['at'])
feat['interest_coverage'] = lev.interest_coverage(df['ebitda'], df['xint'])
feat['st_debt_share']     = lev.st_debt_share(df['dlc'], total_debt)

# Profitability
feat['roa']                = prof.roa(df['ni'], df['at'])
feat['roe']                = prof.roe(df['ni'], df['ceq'])
feat['ebitda_margin']      = prof.ebitda_margin(df['ebitda'], df['sale'])
feat['gross_margin']       = prof.gross_margin(df['gp'], df['sale'])
feat['operational_margin'] = prof.operational_margin(df['oiadp'], df['sale'])

# Liquidity
wc = df['wcap'].where(df['wcap'].notna(), df['act'] - df['lct'])
feat['current_ratio']  = liq.current_ratio(df['act'], df['lct'])
feat['quick_ratio']    = liq.quick_ration(df['che'], df['rect'], df['lct'])
feat['cash_to_assets'] = liq.cash_to_assets(df['che'], df['at'])
feat['wc_to_assets']   = wc / df['at']

# Size
def safe_log(s): return np.log(s.where(s > 0))
feat['log_assets']  = safe_log(df['at'])
feat['log_revenue'] = safe_log(df['sale'])
feat['log_mktcap']  = safe_log(df['csho'] * df['prcc_f'])
feat['emp']         = df['emp']

# Activity
feat['asset_turnover']       = act.asset_turnover(df['sale'], df['at'])
feat['receivables_turnover'] = act.receivables_turnover(df['sale'], df['rect'])
feat['inventory_turnover']   = act.inventory_turnover(df['cogs'], df['invt'])

# Growth — at quarterly frequency, use lag(4) for year-over-year same-quarter comparison
g = df.groupby('gvkey', sort=False)
feat['revenue_growth'] = grow.revenue_growth(df['sale'], g['sale'].shift(4))
feat['asset_growth']   = grow.asset_growth(df['at'],   g['at'].shift(4))
feat['emp_growth']     = np.nan  # no quarterly emp

# Cash flow
feat['opcf_to_assets']  = cf.opcf_to_assets(df['oancf'], df['at'])
feat['capex_to_assets'] = cf.capex_to_assets(df['capx'], df['at'])
feat['fcf_to_assets']   = cf.fcf_to_assets(df['oancf'], df['capx'], df['at'])
print('accounting features computed')

### EBIT fallback + Altman Z routing
Same logic as annual. With `ebit` unavailable in quarterly, every observation falls through to either `oiadp_ttm` or (if missing) `pi_ttm + xint_ttm`. Routing rules are unchanged.

In [ ]:
# EBIT fallback ebit -> oiadp -> pi+xint
ebit_proxy = df['ebit'].where(df['ebit'].notna(), df['oiadp'])
ebit_proxy = ebit_proxy.where(ebit_proxy.notna(), df['pi'].add(df['xint'], fill_value=np.nan))
feat['ebit_proxy'] = ebit_proxy
print(f'ebit_proxy non-null: {ebit_proxy.notna().sum():,} / {len(ebit_proxy):,} '
      f'({ebit_proxy.notna().mean():.1%})')

In [ ]:
v_z   = np.vectorize(cs.altman_z,              otypes=[float])
v_zp  = np.vectorize(cs.altman_z_prime,        otypes=[float])
v_zpp = np.vectorize(cs.altman_z_double_prime, otypes=[float])

sic_s      = df['sic'].astype('Int64')
is_fin     = sic_s.between(6000, 6999).fillna(False)
is_mfr     = sic_s.between(2000, 3999).fillna(False)
has_market = df['csho'].notna() & df['prcc_f'].notna()

altman_z_score = np.full(len(df), np.nan)
altman_variant = np.full(len(df), '', dtype=object)

m = (is_mfr & has_market & ~is_fin).values
if m.any():
    altman_z_score[m] = v_z(
        df.loc[m, 'act'].values, df.loc[m, 'lct'].values,
        df.loc[m, 're'].values,  ebit_proxy[m].values,
        df.loc[m, 'csho'].values, df.loc[m, 'prcc_f'].values,
        df.loc[m, 'lt'].values,  df.loc[m, 'sale'].values,
        df.loc[m, 'at'].values,  df.loc[m, 'wcap'].values,
    )
    altman_variant[m] = 'z'
m = (is_mfr & ~has_market & ~is_fin).values
if m.any():
    altman_z_score[m] = v_zp(
        df.loc[m, 'act'].values, df.loc[m, 'lct'].values,
        df.loc[m, 're'].values,  ebit_proxy[m].values,
        df.loc[m, 'ceq'].values, df.loc[m, 'lt'].values,
        df.loc[m, 'sale'].values, df.loc[m, 'at'].values,
        df.loc[m, 'wcap'].values,
    )
    altman_variant[m] = 'z_prime'
m = (~is_mfr & ~is_fin).values
if m.any():
    altman_z_score[m] = v_zpp(
        df.loc[m, 'act'].values, df.loc[m, 'lct'].values,
        df.loc[m, 're'].values,  ebit_proxy[m].values,
        df.loc[m, 'ceq'].values, df.loc[m, 'lt'].values,
        df.loc[m, 'at'].values,  df.loc[m, 'wcap'].values,
    )
    altman_variant[m] = 'z_double_prime'
altman_variant[is_fin.values] = 'financial_excluded'

feat['altman_z']       = altman_z_score
feat['altman_variant'] = altman_variant

zone = np.full(len(df), '', dtype=object)
for var in ('z', 'z_prime', 'z_double_prime'):
    sel = (altman_variant == var)
    if sel.any():
        zone[sel] = [cs.z_zone(v, var) if not np.isnan(v) else np.nan
                     for v in altman_z_score[sel]]
feat['altman_zone'] = zone

print('Altman variant breakdown:')
print(feat['altman_variant'].value_counts(dropna=False).to_string())

## 4. CRSP trailing market features
Per (`gvkey`, `datadate`):
- `ret_3m`, `ret_12m`: cumulative monthly return over trailing 3 / 12 months — computed as `exp(sum(log1p(RET))) - 1` for numerical stability and exact compounding.
- `volatility_12m`: std dev of trailing 12 monthly `RET`.
- `avg_volume`: trailing-12m mean of `VOL`.
- `share_turnover`: trailing-12m mean of `VOL / SHROUT`.
- `log_mktcap_crsp`: `log(abs(PRC) * SHROUT)` at the matched month-end.
- `market_to_book`: market cap (in $millions = `abs(PRC) * SHROUT / 1000`) divided by `ceq` (also $millions).

`abs(PRC)` because CRSP encodes negative prices when reporting bid/ask midpoint. Firms with `has_crsp == False` (no high-confidence Compustat–CRSP link) get NaN for all market features by construction.

In [ ]:
stamp('loading CRSP monthly...')
crsp = pd.read_csv(
    PROJECT_ROOT / 'data/raw/crsp/CRSP.csv',
    usecols=['PERMNO', 'date', 'RET', 'PRC', 'VOL', 'SHROUT'],
    dtype={'PERMNO': 'Int64'},
    low_memory=False,
)
crsp['date'] = pd.to_datetime(crsp['date'])
# RET sometimes encodes 'C', 'B', 'A' for missing — coerce to numeric NaN
crsp['RET'] = pd.to_numeric(crsp['RET'], errors='coerce')
crsp = crsp.sort_values(['PERMNO', 'date']).reset_index(drop=True)
stamp(f'crsp raw: {len(crsp):,} rows')

In [ ]:
# Per-permno rolling features
crsp['mktcap']   = crsp['PRC'].abs() * crsp['SHROUT']        # $thousands
crsp['turnover'] = crsp['VOL'] / crsp['SHROUT']
crsp['logret']   = np.log1p(crsp['RET'])

g = crsp.groupby('PERMNO', sort=False)
crsp['ret_12m']         = np.exp(g['logret'].rolling(12, min_periods=12).sum()
                                  .reset_index(level=0, drop=True)) - 1
crsp['ret_3m']          = np.exp(g['logret'].rolling(3,  min_periods=3).sum()
                                  .reset_index(level=0, drop=True)) - 1
crsp['volatility_12m']  = (g['RET'].rolling(12, min_periods=12).std()
                              .reset_index(level=0, drop=True))
crsp['avg_volume']      = (g['VOL'].rolling(12, min_periods=12).mean()
                              .reset_index(level=0, drop=True))
crsp['share_turnover']  = (g['turnover'].rolling(12, min_periods=12).mean()
                              .reset_index(level=0, drop=True))
crsp['log_mktcap_crsp'] = np.log(crsp['mktcap'].where(crsp['mktcap'] > 0))
stamp('rolling features computed')

In [ ]:
fu = (pd.read_parquet(PROJECT_ROOT / 'data/clean/firm_universe.parquet')
        .reset_index()[['gvkey', 'permno', 'has_crsp']])
feat = feat.merge(fu, on='gvkey', how='left')
feat['permno']   = feat['permno'].astype('Int64')
# gvkeys missing from firm_universe (failed Phase 1 filters) get has_crsp = False
feat['has_crsp'] = feat['has_crsp'].fillna(False).astype(bool)

crsp_keep = (crsp[['PERMNO', 'date', 'PRC', 'SHROUT',
                   'ret_12m', 'ret_3m', 'volatility_12m',
                   'avg_volume', 'share_turnover', 'log_mktcap_crsp']]
             .rename(columns={'PERMNO': 'permno', 'date': 'crsp_date'})
             .sort_values(['crsp_date', 'permno']))

feat_eligible = (feat[feat['has_crsp'] & feat['permno'].notna()]
                 .sort_values(['datadate', 'permno'])
                 .reset_index(drop=True))

stamp(f'asof merge: {len(feat_eligible):,} crsp-eligible firm-quarters')
m = pd.merge_asof(
    feat_eligible[['gvkey', 'datadate', 'permno']],
    crsp_keep,
    by='permno',
    left_on='datadate',
    right_on='crsp_date',
    direction='backward',
    tolerance=pd.Timedelta(days=45),
)
stamp(f'asof matched: {m["crsp_date"].notna().sum():,}')

In [ ]:
m_features = ['ret_12m', 'ret_3m', 'volatility_12m',
              'avg_volume', 'share_turnover', 'log_mktcap_crsp', 'PRC', 'SHROUT']
feat = feat.merge(m[['gvkey', 'datadate'] + m_features], on=['gvkey', 'datadate'], how='left')

# market_to_book: mktcap in $M = abs(PRC) * SHROUT / 1000  ;  ceq is also $M
feat['market_to_book'] = (feat['PRC'].abs() * feat['SHROUT'] / 1000.0) / df['ceq'].values
feat = feat.drop(columns=['PRC', 'SHROUT'])
print(f'ret_12m non-null: {feat["ret_12m"].notna().sum():,} '
      f'({feat["ret_12m"].notna().mean():.1%})')

## 5. FRED macro merge + S&P 500 trailing return
As-of backward merge: each firm-quarter gets the most recent FRED observation at-or-before its `datadate`. Adds 14 FRED series (the FRED `SP500` column is dropped — its 16.8% coverage is too sparse, since FRED's daily SP500 series only retains ~10 years of history) plus the derived `BAA_AAA_spread = DBAA - DAAA`.

Then attaches **`sp500_ret_12m`** — trailing 12-month S&P 500 return computed from `data/raw/sp500/sp500_monthly.csv` (yfinance ^GSPC daily resampled to month-start, full history 1980-01-01 → 2024-12-01).

In [ ]:
# --- FRED macro merge (drops the FRED SP500 column — only ~16.8% coverage) ---
# tolerance=45d so post-series-end firm-quarters get NaN rather than stale carry-forward.
fred = pd.read_csv(PROJECT_ROOT / 'data/raw/fred/fred_master.csv')
fred['date'] = pd.to_datetime(fred['date'])
fred = fred.drop(columns=['SP500']).sort_values('date').reset_index(drop=True)
fred['BAA_AAA_spread'] = fred['DBAA'] - fred['DAAA']

feat = feat.sort_values('datadate').reset_index(drop=True)
feat = pd.merge_asof(feat, fred, left_on='datadate', right_on='date',
                     direction='backward', tolerance=pd.Timedelta(days=45))
feat = feat.drop(columns=['date'])
stamp(f'fred attached (no SP500). VIXCLS non-null: {feat["VIXCLS"].notna().sum():,} '
      f'({feat["VIXCLS"].notna().mean():.1%})')

# --- yfinance S&P 500 trailing 12-month return (replaces FRED SP500) ---
sp500 = pd.read_csv(PROJECT_ROOT / 'data/raw/sp500/sp500_monthly.csv')
sp500['date'] = pd.to_datetime(sp500['date'])
sp500 = sp500.sort_values('date').reset_index(drop=True)
sp500['sp500_ret_12m'] = sp500['close'].pct_change(periods=12)
sp500 = sp500[['date', 'sp500_ret_12m']]

feat = pd.merge_asof(feat, sp500, left_on='datadate', right_on='date',
                     direction='backward', tolerance=pd.Timedelta(days=45))
feat = feat.drop(columns=['date'])
stamp(f'sp500_ret_12m attached: {feat["sp500_ret_12m"].notna().sum():,} non-null '
      f'({feat["sp500_ret_12m"].notna().mean():.1%})')

## 6. Save `node_features_quarterly.parquet`

In [ ]:
# Final inf sweep
num_cols = feat.select_dtypes(include=[np.number]).columns
inf_before = {c: int(np.isinf(feat[c]).sum()) for c in num_cols if np.isinf(feat[c]).any()}
print('inf counts before final sweep:', inf_before or '(none)')
feat[num_cols] = feat[num_cols].replace([np.inf, -np.inf], np.nan)

out_path = Path('node_features_quarterly.parquet')
feat.to_parquet(out_path, index=False)
print(f'wrote {out_path} — {feat.shape[0]:,} rows × {feat.shape[1]} cols, '
      f'{out_path.stat().st_size/1e6:.1f} MB')

## 7. Validation

### 7.1 Shape, gvkey count, date range

In [ ]:
print(f'rows:           {len(feat):,}')
print(f'unique gvkeys:  {feat["gvkey"].nunique():,}')
print(f'date range:     {feat["datadate"].min().date()} → {feat["datadate"].max().date()}')
print(f'quarter counts:')
print(feat['fqtr'].value_counts().sort_index().to_string())

### 7.2 Coverage table — non-null % per feature

In [ ]:
nn = feat.notna().sum().sort_values(ascending=False)
coverage = (nn / len(feat) * 100).round(1)
pd.DataFrame({'non_null': nn, 'coverage_pct': coverage})

### 7.3 Infinity check (post-cleanup)

In [ ]:
num_cols = feat.select_dtypes(include=[np.number]).columns
{c: int(np.isinf(feat[c]).sum()) for c in num_cols if np.isinf(feat[c]).any()} or 'no infinities'

### 7.4 Market features NaN-by-design for firms without CRSP
Every market feature should be NaN for `has_crsp == False` rows.

In [ ]:
no_crsp = feat[~feat['has_crsp']]
print(f'rows without CRSP link: {len(no_crsp):,}')
mkt_features = ['ret_12m', 'ret_3m', 'volatility_12m', 'avg_volume',
                'share_turnover', 'log_mktcap_crsp', 'market_to_book']
pd.DataFrame({
    'feature': mkt_features,
    'non_null_in_no_crsp': [int(no_crsp[c].notna().sum()) for c in mkt_features],
    'expected': 0,
})

### 7.5 Spot-check: Enron quarterly trajectory leading to bankruptcy
Enron filed Chapter 11 on **Dec 2, 2001**. Last fiscal-quarter datadate in our universe is Q3-2001 (2001-09-30). The trajectory should show: stable distress Z (1998–2000) → bubble M/B blow-out → sharp Z and return collapse in 2001.

In [ ]:
en = feat[feat['gvkey'] == 6127].sort_values('datadate')
view = ['datadate', 'fyearq', 'fqtr', 'altman_z', 'altman_zone',
        'debt_to_assets', 'roa', 'log_assets',
        'ret_12m', 'ret_3m', 'volatility_12m', 'market_to_book']
en[view].tail(16)

### 7.6 Diagnostic — what dropped Enron's Z to 0.19 in Q3-2001?
Compare the 2000-Q4 (peak) and 2001-Q3 (collapse) snapshots side by side.

In [ ]:
key_inputs = ['act', 'lct', 're', 'oiadp', 'pi', 'xint', 'csho', 'prcc_f',
              'lt', 'sale', 'at', 'wcap', 'ceq']
en_inputs = (df[df['gvkey'] == 6127]
             [['datadate', 'fyearq', 'fqtr'] + key_inputs]
             .sort_values('datadate'))
focus = en_inputs[en_inputs['datadate'].isin(['2000-12-31', '2001-09-30'])].set_index('datadate')
focus.T